# Project 22 — Breadth-First Search: Real Social-Network Recommendation Engine

**Business problem:** efficiently discover how people are connected and generate explainable second-degree connection candidates.

This project uses the **Stanford SNAP Facebook combined ego-network** rather than a toy graph. It implements BFS from scratch, verifies it against NetworkX, measures degrees of separation, creates friends-of-friends recommendations, visualises an actual search path and benchmarks query latency.

**Big-company connection:** this is directly analogous to the graph candidate-generation layer behind products such as LinkedIn's *People You May Know*: members are nodes, connections are edges, and second-degree neighbours are high-signal candidates before a downstream ranking model.

**Algorithmic complexity:** BFS runs in **O(V + E)** time and **O(V)** auxiliary space for an adjacency-list graph.

In [ ]:
import io, gzip, time, requests
from collections import deque
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

SEED = 42
SNAP_URL = "https://snap.stanford.edu/data/facebook_combined.txt.gz"

r = requests.get(SNAP_URL, timeout=60)
r.raise_for_status()
with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
    edges = pd.read_csv(gz, sep=" ", names=["source", "target"])

G = nx.from_pandas_edgelist(edges, "source", "target")
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"Connected components: {nx.number_connected_components(G)}")
print(f"Average degree: {np.mean([d for _, d in G.degree()]):.2f}")

## Custom BFS — shortest path and degrees of separation

Unlike DFS, BFS explores the graph **level by level**, so on an unweighted network the first time it reaches a target is guaranteed to be a shortest path.

In [ ]:
def bfs_shortest_path(graph, start, goal):
    if start not in graph or goal not in graph:
        return None, 0
    queue = deque([start])
    parent = {start: None}
    expanded = 0

    while queue:
        node = queue.popleft()
        expanded += 1
        if node == goal:
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1], expanded

        for nbr in graph.neighbors(node):
            if nbr not in parent:
                parent[nbr] = node
                queue.append(nbr)

    return None, expanded

# Reproducible real example: select two well-connected users
degree_rank = sorted(G.degree, key=lambda x: x[1], reverse=True)
viewer = degree_rank[0][0]
target = degree_rank[25][0]

path, expanded = bfs_shortest_path(G, viewer, target)
nx_path = nx.shortest_path(G, viewer, target)

print("Viewer:", viewer, "| target:", target)
print("BFS path:", path)
print("Degrees of separation:", len(path) - 1)
print("Nodes expanded:", expanded)
print("Matches NetworkX shortest path length:", len(path) == len(nx_path))

## LinkedIn-style second-degree candidate generation

A recommendation engine should not score every possible user. A practical graph stage first retrieves a smaller candidate pool: **friends of friends who are not already connected**.

In [ ]:
direct = set(G.neighbors(viewer))
second_degree = set()

for friend in direct:
    second_degree.update(G.neighbors(friend))

second_degree.discard(viewer)
second_degree -= direct

rows = []
for candidate in second_degree:
    mutual = len(list(nx.common_neighbors(G, viewer, candidate)))
    union = len(set(G.neighbors(viewer)) | set(G.neighbors(candidate)))
    jaccard = mutual / union if union else 0.0
    rows.append((candidate, mutual, jaccard, G.degree(candidate)))

recs = (pd.DataFrame(rows, columns=["candidate","mutual_connections","jaccard","degree"])
          .sort_values(["mutual_connections","jaccard","degree"], ascending=False)
          .head(10)
          .reset_index(drop=True))

print(f"Viewer {viewer} has {len(direct):,} direct connections")
print(f"Eligible second-degree candidates: {len(second_degree):,}")
display(recs)

In [ ]:
# Explain the top recommendation with an actual BFS path
top_candidate = int(recs.iloc[0]["candidate"])
top_path, top_expanded = bfs_shortest_path(G, viewer, top_candidate)

print("TOP RECOMMENDATION")
print(f"{viewer} -> {top_candidate}")
print("Shortest connection path:", " -> ".join(map(str, top_path)))
print("Degrees of separation:", len(top_path)-1)
print("Mutual connections:", int(recs.iloc[0]["mutual_connections"]))

In [ ]:
# Visualise the real shortest path and surrounding graph
path_nodes = set(top_path)
context = set(path_nodes)
for n in top_path:
    context.update(list(G.neighbors(n))[:8])

H = G.subgraph(context).copy()
pos = nx.spring_layout(H, seed=SEED)

plt.figure(figsize=(11, 8))
nx.draw_networkx_edges(H, pos, alpha=0.18)
nx.draw_networkx_nodes(H, pos, node_size=90, alpha=0.75)
nx.draw_networkx_nodes(H, pos, nodelist=top_path, node_size=430)
nx.draw_networkx_labels(H, pos, labels={n:str(n) for n in top_path}, font_size=8)
path_edges = list(zip(top_path[:-1], top_path[1:]))
nx.draw_networkx_edges(H, pos, edgelist=path_edges, width=3)
plt.title("BFS shortest connection path in the real SNAP social graph")
plt.axis("off")
plt.show()

## BFS latency benchmark

In [ ]:
rng = np.random.default_rng(SEED)
nodes = np.array(list(G.nodes()))
pairs = [tuple(rng.choice(nodes, 2, replace=False)) for _ in range(100)]

latency_ms, hops, expanded_counts = [], [], []
for a, b in pairs:
    t0 = time.perf_counter()
    p, ex = bfs_shortest_path(G, int(a), int(b))
    latency_ms.append((time.perf_counter()-t0)*1000)
    hops.append(len(p)-1 if p else np.nan)
    expanded_counts.append(ex)

print(f"Median BFS latency: {np.median(latency_ms):.3f} ms")
print(f"95th percentile latency: {np.percentile(latency_ms,95):.3f} ms")
print(f"Mean observed separation: {np.nanmean(hops):.2f}")
print(f"Mean nodes expanded: {np.mean(expanded_counts):,.0f}")

## Interview takeaway

**Use BFS when the graph is unweighted and you need minimum-hop reachability.** In a large social platform, BFS-like neighbourhood expansion is a natural way to retrieve second-degree candidates and explain *how* two members are connected.

**CV bullet:** Built an explainable social-network recommendation prototype on a real 4,039-node / 88,234-edge Stanford SNAP graph using custom BFS, shortest-path reconstruction, friends-of-friends candidate generation, mutual-connection ranking, graph visualisation and latency benchmarking.